In [77]:
import operator
import random
from typing import Annotated, TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class InterruptibleState(TypedDict):
    messages: Annotated[list, operator.add]
    current_step: str


def print_node(state: InterruptibleState) -> InterruptibleState:
    print("模拟正常节点")
    return {"messages": ["模拟正常节点"], "current_step": "pending"}


def risky_step(state: InterruptibleState) -> InterruptibleState:
    if random.choice([True, False]):
        raise Exception("模拟随机错误！工作流中断！")
    print("随机节点正常")
    return {"messages": ["成功完成危险步骤！"], "current_step": "completed"}


builder = StateGraph(InterruptibleState)
builder.add_node("risky_step", risky_step)
builder.add_node("print_node", print_node)
builder.add_edge(START, "print_node")
builder.add_edge("print_node", "risky_step")
builder.add_edge("risky_step", END)

graph = builder.compile(checkpointer=InMemorySaver())


In [88]:
state = {"messages": ["开始"], "current_step": "start"}
import uuid

config: RunnableConfig = {"configurable": {"thread_id": str(uuid.uuid4())}}
while True:
    try:
        result = graph.invoke(state, config=config)
        print("finish")
        break
    except Exception:
        state = None
        print("error")

模拟正常节点
error
error
error
随机节点正常
finish
